# 01 — Pipeline Limpo (2016–2024)

Constrói o corpus rotulado do TCU com **feature única VOTO_LIMPO** (sem vazamento).
Nenhuma célula neste notebook cria features a partir do SUMARIO.

**Saída (persistida no Google Drive quando montado — sobrevive ao fim da sessão):**
- `data/interim/acordaos_rotulados.parquet` — corpus 2016–2024 rotulado.
- `data/processed/{train,val,test}.parquet` — splits (temporal por padrão).
- `resultados/metricas_pipeline.json` — sumário do corpus e auditoria de vazamento.

Sem Google Drive montado, as saídas ficam no clone local do repositório e são
perdidas quando o runtime do Colab é reiniciado.


## 1. Setup


In [ ]:
import os, sys, subprocess
REPO_DIR = os.environ.get('REPO_DIR', '/content/deep-acordao-tcu2')
REPO_URL = 'https://github.com/bsousa7/deep-acordao-tcu2.git'
BRANCH = os.environ.get('BRANCH', 'claude/deep-acordao-tcu-refactor-yyjfr3')
if not os.path.isdir(os.path.join(REPO_DIR, 'src')):
    subprocess.run(['git', 'clone', REPO_URL, '--branch', BRANCH, REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('repo em', REPO_DIR)


In [ ]:
%pip -q install pandas pyarrow scikit-learn scipy nltk requests tqdm


## 2. Configuração — escopo temporal 2016–2024


In [ ]:
from pathlib import Path
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

ANOS = list(range(2016, 2025))         # 2016..2024 (9 anos)
ANO_VAL = 2023
ANO_TESTE = 2024
RANDOM_STATE = 42

BASE = Path(REPO_DIR)
DATA_RAW = BASE / 'data' / 'raw'       # CSVs de origem — symlink do Drive (célula seguinte)

# --------------------------------------------------------------
# Persistência no Google Drive
# --------------------------------------------------------------
# data/interim, data/processed e resultados/ passam a viver em
# /content/drive/MyDrive/deep-acordao-tcu2/ quando o Drive está montado —
# sobrevivem ao reinício do runtime e ficam disponíveis para os notebooks
# 02 e 03 mesmo em uma sessão totalmente nova. Sem Drive, cai para o clone
# local (perdido ao fim da sessão).
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive') / 'deep-acordao-tcu2'
except Exception as _e:
    DRIVE_ROOT = None
    print(f'Google Drive indisponível (fora do Colab?): {_e}')

PERSIST_BASE = DRIVE_ROOT if DRIVE_ROOT is not None else BASE
DATA_INTERIM = PERSIST_BASE / 'data' / 'interim'
DATA_PROCESSED = PERSIST_BASE / 'data' / 'processed'
RESULTADOS = PERSIST_BASE / 'resultados'

for d in [DATA_RAW, DATA_INTERIM, DATA_PROCESSED, RESULTADOS / 'figuras']:
    d.mkdir(parents=True, exist_ok=True)

print(f'Anos alvo: {ANOS}')
print(f'Persistindo data/interim, data/processed e resultados/ em: {PERSIST_BASE}')


## 3. CSVs anuais — Google Drive (primário) + download TCU (fallback)

Os CSVs anuais pesam ~200–500 MB cada. No Colab, o mais barato é mantê-los no seu
Google Drive e apenas apontá-los para `data/raw/` via *symlink* — não copia bytes.

Fluxo desta célula:
1. Monta o Google Drive (se ainda não estiver montado).
2. Para cada ano de 2016..2024, procura `acordao-completo-{ano}.csv` em
   `/content/drive/MyDrive/deep-acordao-tcu2/data/raw/` (ajuste `DRIVE_RAW` se
   guardar em outra pasta) e cria um symlink em `data/raw/`.
3. Se algum ano faltar no Drive, tenta baixar do portal TCU como fallback.


In [ ]:
from src.aquisicao.baixar_csvs import baixar_todos

# Reaproveita o Drive já montado na célula de configuração (DRIVE_ROOT).
DRIVE_RAW = (DRIVE_ROOT / 'data' / 'raw') if DRIVE_ROOT is not None else None
if DRIVE_RAW is not None:
    print(f'Procurando CSVs em: {DRIVE_RAW}')
    if not DRIVE_RAW.exists():
        print('  (pasta ainda não existe no Drive — nenhum CSV será encontrado por aqui)')

TAM_MIN = 50_000_000

def _apontar_para_drive(ano: int) -> Path | None:
    destino = DATA_RAW / f'acordao-completo-{ano}.csv'
    if destino.exists() and destino.stat().st_size >= TAM_MIN:
        return destino
    if DRIVE_RAW is None:
        return None
    origem = DRIVE_RAW / f'acordao-completo-{ano}.csv'
    if not origem.exists() or origem.stat().st_size < TAM_MIN:
        return None
    if destino.exists() or destino.is_symlink():
        destino.unlink()
    try:
        destino.symlink_to(origem)
    except OSError:
        # Se symlink não for permitido no FS, copia (fallback lento)
        import shutil
        shutil.copy2(origem, destino)
    return destino

presentes, faltantes = {}, []
for ano in ANOS:
    p = _apontar_para_drive(ano)
    if p is not None and p.exists() and p.stat().st_size >= TAM_MIN:
        presentes[ano] = p
        origem = 'Drive' if p.is_symlink() else 'local'
        print(f'  {ano}: {p.stat().st_size/1e6:5.0f} MB  ✓ ({origem})')
    else:
        faltantes.append(ano)
        print(f'  {ano}: ausente')

# --------------------------------------------------------------
# Fallback — baixa do portal TCU apenas o que falta no Drive
# --------------------------------------------------------------
if faltantes:
    print(f'\nBaixando do TCU (não encontrados no Drive): {faltantes}')
    baixar_todos(faltantes, data_dir=DATA_RAW)

arquivos = {a: DATA_RAW / f'acordao-completo-{a}.csv'
            for a in ANOS
            if (DATA_RAW / f'acordao-completo-{a}.csv').exists()}
print(f'\nAnos disponíveis: {sorted(arquivos)}')
assert arquivos, 'Nenhum CSV disponível em data/raw/ nem no Google Drive.'


## 4. Filtro temático + rotulagem


In [ ]:
from src.preprocessamento.filtrar_tematico import combinar_anos, salvar_parquet

df = combinar_anos(sorted(arquivos.keys()), data_dir=DATA_RAW, apenas_tema=True)
print('n =', len(df))
print(df['LABEL'].value_counts().to_string())
print()
print('Distribuição por ano:')
print(df.groupby(['ANO', 'LABEL']).size().unstack(fill_value=0).to_string())


## 5. Construção da feature única — `VOTO_LIMPO`

Remove o dispositivo do VOTO e mascara termos de veredito residuais. Nenhuma feature
é derivada do SUMARIO. O gate de auditoria exige fração de vazamento = 0.


In [ ]:
from src.preprocessamento.anti_vazamento import construir_feature_voto, auditar_vazamento, contem_veredito

feats = df['VOTO'].apply(construir_feature_voto)
df['VOTO_LIMPO'] = feats.apply(lambda f: f.texto)
df['dispositivo_encontrado'] = feats.apply(lambda f: f.dispositivo_encontrado)

print(f'Dispositivo localizado em {100*df["dispositivo_encontrado"].mean():.1f}% dos votos')

# Auditoria comparativa (SUMARIO só para diagnóstico — jamais será usado como feature)
aud_sumario = auditar_vazamento(df, 'SUMARIO', verbose=True)
aud_voto_limpo = auditar_vazamento(df, 'VOTO_LIMPO', verbose=True)
assert aud_voto_limpo['gate_passou'], 'Gate de vazamento FALHOU em VOTO_LIMPO — investigar padrões de dispositivo.'
print('GATE VOTO_LIMPO: OK — feature honesta para treino.')


## 6. Filtro de tamanho mínimo e persistência


In [ ]:
N_MIN_CHARS = 200
n_antes = len(df)
df = df[df['VOTO_LIMPO'].str.len() >= N_MIN_CHARS].reset_index(drop=True)
print(f'Após filtro >= {N_MIN_CHARS} chars: {len(df)} (removidos {n_antes - len(df)})')

salvar_parquet(df, DATA_INTERIM / 'acordaos_rotulados.parquet')


## 7. Split — temporal (padrão) ou estratificado (fallback)

**Temporal (recomendado):** treino ≤ 2022, val = 2023, teste = 2024. Espelha o uso real
e evita 'ver o futuro'.

**Estratificado 70/15/15:** fallback para corpora pequenos por classe.


In [ ]:
from collections import Counter
from src.preprocessamento.split_temporal import (
    dividir_temporal_por_ano, dividir_estratificado, salvar_splits,
)

# Verifica cobertura mínima por classe/ano antes de decidir
cobertura = df.groupby(['ANO', 'LABEL']).size().unstack(fill_value=0)
print(cobertura)

usar_temporal = ANO_VAL in df['ANO'].unique() and ANO_TESTE in df['ANO'].unique()
if usar_temporal:
    train_df, val_df, test_df = dividir_temporal_por_ano(df, ano_teste=ANO_TESTE, ano_val=ANO_VAL)
    tipo_split = 'temporal'
else:
    train_df, val_df, test_df = dividir_estratificado(df)
    tipo_split = 'estratificado'

salvar_splits(train_df, val_df, test_df, DATA_PROCESSED)
print(f'\nSplit tipo = {tipo_split}')
print(f'treino={len(train_df)} val={len(val_df)} teste={len(test_df)}')


## 8. Sumário e persistência


In [ ]:
from src.avaliacao.metricas import salvar_json

sumario = {
    'escopo': {
        'anos': ANOS,
        'anos_disponiveis': sorted(arquivos.keys()),
        'ano_val': ANO_VAL,
        'ano_teste': ANO_TESTE,
        'tipo_split': tipo_split,
    },
    'corpus': {
        'total_apos_filtro_tematico_e_tamanho': int(len(df)),
        'distribuicao_global': df['LABEL'].value_counts().to_dict(),
        'distribuicao_por_ano': df.groupby(['ANO','LABEL']).size().unstack(fill_value=0).to_dict(),
        'dispositivo_localizado_frac': float(df['dispositivo_encontrado'].mean()),
    },
    'auditoria_vazamento': {
        'SUMARIO_referencia': aud_sumario,
        'VOTO_LIMPO_feature': aud_voto_limpo,
    },
    'splits': {
        'treino': {'n': int(len(train_df)), 'dist': train_df['LABEL'].value_counts().to_dict()},
        'val':    {'n': int(len(val_df)),   'dist': val_df['LABEL'].value_counts().to_dict()},
        'teste':  {'n': int(len(test_df)),  'dist': test_df['LABEL'].value_counts().to_dict()},
    },
}
salvar_json(sumario, RESULTADOS / 'metricas_pipeline.json')
print(f'OK — dados persistidos em: {PERSIST_BASE}')
print('Prontos para 02_baseline_ponderado.ipynb e 03_textcnn_ponderado.ipynb'
      ' (inclusive em uma sessão nova, desde que o Drive esteja montado).')
